From the youtube videos class. Since I am trying to solve a game that is played in a space like battleship I think that taking his heatmap idea would be a great idea. My hypothesis for how it would look is the star position would be the most played first move with the 3,4 and 4,3 in each corner being next if not the star point. The order would mostlikely be Corner --> Edge --> Fight where there is competition --> Center. This hypothesis would only work for the 19x19 because I really only know that board compared to the 9x9 and 11x11 boards.

For a given strategy I would try to identify something similar to "influence" in a given space and assign it a vector/direction like a vertical wall will have influence to the horizontal sides of it where there are little stones fromn the other player. I should do some research as to how much influence a single piece has alone and where it has more or less on an empty board.

For a strategy of Go. To make it simple I will create one of a few choices where you analyze the board state based on the last move played by the other player. Baseline is that players will place a stone to increase their influence and liberties while decreasing your influence and liberties. Think in micro vs. macro moves always asses the current fights on the board and see if you are up or down liberties compared to the other opponent in each fight and prioritize fights with the most influence then pieces at risk.

Board reduction: Symmetry of the board in rotation


Lecturing about connect4:
You need a clean way of representing your state so then your reward function can get you more uses in more states

Reward Function(State, Action). Example: 4(W) in a row gets tons of points, 3(W) + Empty gets slightly less, 2(w) + Empty gets slightly less, 4(B) gets large negative points, 3(B) + Empty gets slighlty less negative, etc. To evaluate a move you can make computer go 3 moves in the future

My first strategy for a reward function for GO of specifically 9x9: Agressive

Rewards:

Decreasing amount of points starting with the middle
If captures opponent pieces gets the most points
Rid of liberty of opponent and move is closest to same color massive points
Creates a connected structure of two eyes medium amount of points
Placeing piece in a corner spot will give large negative points
Placing a piece on board of board will give medium negtative points



I'm building a 9x9 Go agent in Python for a computational economics class, and I need a reward function that scores candidate moves, which I'll keep changing as I experiment. [Paste your board code here, or say you need a minimal Board class with liberty and capture logic.] My first strategy, "Aggressive," has six rules. Moves earn a bonus that shrinks with distance from the center point (4,4). Capturing opponent stones earns the largest reward, scaled by the number of stones captured. A move that removes an opponent group's liberty while sitting next to one of my own stones earns a massive reward. Creating a connected group with two eyes earns a medium reward. Corner moves get a large penalty, and other edge moves get a medium penalty. Before coding, ask me whether captures or the liberty-and-connection rule should rank higher, and whether corner moves should get both penalties. Write each rule as its own small function, and keep all weights in a separate config (a dict or YAML file), so I can change numbers, toggle rules on or off, or add new strategies without editing the logic. Make `compute_reward(board, move, color, strategy)` return both the total and a per-rule breakdown so I can debug scores. Handle illegal moves clearly, and leave a hook for a final win/loss reward. Include pytest tests that use small hand-built positions to check each rule. Finally, show me the score breakdown for three example moves so I can sanity-check the weights.

## Engine update

What changed from the first version:
- **Ko rule added.** The old code let both players retake the same stone forever. (Your test game shows it: White 3 4, Black 3 3, White 3 4, ...) Test 4 below replays that exact game.
- **`play_move` returns a `MoveResult`.** It still works as True/False, and it also gives the reason a move was illegal (`"suicide"`, `"ko"`, ...) and the list of **captured stones**, which the reward function needs.
- **Atari and eye helpers:** `is_in_atari`, `groups_in_atari`, `is_eye`, `count_eyes`, `has_two_eyes`.
- **Solver tools:** `copy()`, `is_legal()` (checks a move without playing it), `legal_moves()`.
- **Scoring:** `score()` and `winner()` use area scoring with komi 7.5, which fills in your old TODO.
- **Faster:** neighbors are precomputed, and the group and its liberties come from one search. Illegal moves are undone without copying the whole board. In random games this runs about 7–8x faster than the first version.
- **No `print` inside the engine**, so a solver can test thousands of moves silently.


In [1]:
# =========================================================
# 9x9 Go engine
#
# Rules implemented:
#   - stones on intersections, groups = orthogonally connected
#   - liberties, capture, atari
#   - suicide is illegal unless the move captures
#   - ko: a move may not recreate the previous whole-board state
#     (optional positional superko: never repeat ANY earlier state)
#   - passing; two passes in a row end the game
#   - eye detection + "two eyes" check (for the reward function)
#   - area scoring (stones + surrounded territory + komi)
# =========================================================

import copy
from dataclasses import dataclass, field

BOARD_SIZE = 9
KOMI = 7.5          # common 9x9 komi; the .5 means no ties

EMPTY = "."
BLACK = "B"
WHITE = "W"


def other_color(color):
    return WHITE if color == BLACK else BLACK


def build_neighbor_table(size):
    """Precompute the orthogonal neighbors of every point once,
    so we never re-check board edges during play (big speed-up)."""
    table = {}
    for r in range(size):
        for c in range(size):
            table[(r, c)] = tuple(
                (r + dr, c + dc)
                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1))
                if 0 <= r + dr < size and 0 <= c + dc < size
            )
    return table


def build_diagonal_table(size):
    """Diagonal neighbors, used only for eye detection."""
    table = {}
    for r in range(size):
        for c in range(size):
            table[(r, c)] = tuple(
                (r + dr, c + dc)
                for dr, dc in ((-1, -1), (-1, 1), (1, -1), (1, 1))
                if 0 <= r + dr < size and 0 <= c + dc < size
            )
    return table


@dataclass
class MoveResult:
    """What happened when a move was tried.
    Works like True/False in an `if`, but also says WHY a move failed
    and WHICH stones were captured (useful for the reward function)."""
    legal: bool
    reason: str = ""
    captured: list = field(default_factory=list)

    def __bool__(self):
        return self.legal


class GoGame:
    def __init__(self, size=BOARD_SIZE, komi=KOMI, superko=False):
        self.size = size
        self.komi = komi
        self.superko = superko

        self.board = [[EMPTY] * size for _ in range(size)]
        self.neighbors = build_neighbor_table(size)
        self.diagonals = build_diagonal_table(size)

        self.current_player = BLACK          # Black always starts
        self.consecutive_passes = 0
        self.captures = {BLACK: 0, WHITE: 0} # stones each player has captured

        # Ko bookkeeping: the board as it was BEFORE the last move.
        # A new move may not recreate it.
        self.previous_state = None
        # Every position seen so far (only used if superko=True)
        self.history = {self.board_key()}
        self.move_history = []               # list of (color, (row, col) or "pass")

    # -----------------------------------------
    # Basic helpers
    # -----------------------------------------

    def board_key(self):
        """The whole board as one string, e.g. '....B....' -> fast to compare."""
        return "".join("".join(row) for row in self.board)

    def is_on_board(self, row, col):
        return 0 <= row < self.size and 0 <= col < self.size

    def get_neighbors(self, row, col):
        return self.neighbors[(row, col)]

    def other_player(self, player):
        return other_color(player)

    def copy(self):
        """Independent copy of the game -> lets a solver try moves
        without touching the real game."""
        new = copy.copy(self)
        new.board = [row[:] for row in self.board]
        new.captures = dict(self.captures)
        new.history = set(self.history) if self.superko else self.history
        new.move_history = list(self.move_history)
        return new

    # -----------------------------------------
    # Groups and liberties
    # -----------------------------------------

    def get_group_and_liberties(self, row, col):
        """One flood fill that returns (stones, liberties) of the group at (row, col)."""
        color = self.board[row][col]
        if color == EMPTY:
            return set(), set()

        stones = {(row, col)}
        liberties = set()
        stack = [(row, col)]
        board = self.board

        while stack:
            point = stack.pop()
            for nr, nc in self.neighbors[point]:
                value = board[nr][nc]
                if value == EMPTY:
                    liberties.add((nr, nc))
                elif value == color and (nr, nc) not in stones:
                    stones.add((nr, nc))
                    stack.append((nr, nc))

        return stones, liberties

    def get_group(self, row, col):
        return self.get_group_and_liberties(row, col)[0]

    def get_liberties(self, group):
        liberties = set()
        for point in group:
            for nr, nc in self.neighbors[point]:
                if self.board[nr][nc] == EMPTY:
                    liberties.add((nr, nc))
        return liberties

    def all_groups(self, color=None):
        """Every group on the board as a list of (color, stones, liberties)."""
        seen = set()
        groups = []
        for r in range(self.size):
            for c in range(self.size):
                stone = self.board[r][c]
                if stone == EMPTY or (r, c) in seen:
                    continue
                if color is not None and stone != color:
                    continue
                stones, libs = self.get_group_and_liberties(r, c)
                seen |= stones
                groups.append((stone, stones, libs))
        return groups

    # -----------------------------------------
    # Atari
    # -----------------------------------------

    def is_in_atari(self, row, col):
        """True if the stone at (row, col) belongs to a group with exactly 1 liberty."""
        if self.board[row][col] == EMPTY:
            return False
        return len(self.get_group_and_liberties(row, col)[1]) == 1

    def groups_in_atari(self, color):
        """All groups of `color` that have exactly one liberty left."""
        return [(stones, libs) for _, stones, libs in self.all_groups(color) if len(libs) == 1]

    # -----------------------------------------
    # Eyes
    # -----------------------------------------

    def is_eye(self, row, col, color):
        """Is this empty point a (single-point) real eye for `color`?

        1. Every orthogonal neighbor must be a `color` stone.
        2. Diagonal check to rule out FALSE eyes:
           - on the edge or in a corner: NO diagonal may be an enemy stone
           - in the middle: at most ONE diagonal may be an enemy stone
        This is the standard heuristic used by most simple Go programs.
        It does not detect bigger multi-point eyes (e.g. a 3-space eye).
        """
        if self.board[row][col] != EMPTY:
            return False

        for nr, nc in self.neighbors[(row, col)]:
            if self.board[nr][nc] != color:
                return False

        enemy = other_color(color)
        diagonals = self.diagonals[(row, col)]
        enemy_diagonals = sum(1 for dr, dc in diagonals if self.board[dr][dc] == enemy)

        if len(diagonals) < 4:          # edge or corner
            return enemy_diagonals == 0
        return enemy_diagonals <= 1

    def count_eyes(self, group):
        """Number of distinct real eyes touching this group."""
        if not group:
            return 0
        r, c = next(iter(group))
        color = self.board[r][c]
        return sum(1 for lr, lc in self.get_liberties(group) if self.is_eye(lr, lc, color))

    def has_two_eyes(self, group):
        """A group with two or more real eyes can never be captured."""
        return self.count_eyes(group) >= 2

    # -----------------------------------------
    # Playing moves
    # -----------------------------------------

    def _apply_move(self, row, col, player):
        """Place a stone and resolve captures / legality.
        Returns (MoveResult, undo_info). If the move is illegal,
        the board is already restored when this returns."""
        if not self.is_on_board(row, col):
            return MoveResult(False, "outside the board"), None
        if self.board[row][col] != EMPTY:
            return MoveResult(False, "point already occupied"), None

        key_before = self.board_key()
        enemy = other_color(player)
        board = self.board
        board[row][col] = player

        # 1) Capture enemy groups that just lost their last liberty
        captured = []
        for nr, nc in self.neighbors[(row, col)]:
            if board[nr][nc] == enemy:
                stones, libs = self.get_group_and_liberties(nr, nc)
                if not libs:
                    for sr, sc in stones:
                        board[sr][sc] = EMPTY
                    captured.extend(stones)

        # 2) Suicide check (only possible if nothing was captured)
        if not captured:
            _, own_libs = self.get_group_and_liberties(row, col)
            if not own_libs:
                board[row][col] = EMPTY
                return MoveResult(False, "suicide"), None

        # 3) Ko check: the new board may not equal the board before the last move
        key_after = self.board_key()
        if key_after == self.previous_state or (self.superko and key_after in self.history):
            board[row][col] = EMPTY
            for sr, sc in captured:
                board[sr][sc] = enemy
            return MoveResult(False, "ko"), None

        return MoveResult(True, "", captured), (key_before, key_after)

    def play_move(self, row, col):
        """Play a stone for the current player. Returns a MoveResult
        (truthy if legal). Illegal moves leave the game unchanged."""
        player = self.current_player
        result, undo_info = self._apply_move(row, col, player)
        if not result:
            return result

        key_before, key_after = undo_info
        self.previous_state = key_before
        if self.superko:
            self.history.add(key_after)
        self.captures[player] += len(result.captured)
        self.move_history.append((player, (row, col)))
        self.consecutive_passes = 0
        self.current_player = other_color(player)
        return result

    def is_legal(self, row, col, player=None):
        """Check a move without changing the game."""
        player = player or self.current_player
        result, undo_info = self._apply_move(row, col, player)
        if result:
            # undo the trial move
            self.board[row][col] = EMPTY
            enemy = other_color(player)
            for sr, sc in result.captured:
                self.board[sr][sc] = enemy
        return result.legal

    def legal_moves(self, player=None):
        """Every legal (row, col) for `player` (passing is always legal too)."""
        player = player or self.current_player
        return [
            (r, c)
            for r in range(self.size)
            for c in range(self.size)
            if self.board[r][c] == EMPTY and self.is_legal(r, c, player)
        ]

    def pass_turn(self):
        self.consecutive_passes += 1
        self.previous_state = None           # a pass lifts the ko restriction
        self.move_history.append((self.current_player, "pass"))
        self.current_player = other_color(self.current_player)

    def game_over(self):
        return self.consecutive_passes >= 2

    # -----------------------------------------
    # Scoring (area scoring, like Chinese rules)
    #   score = your stones on the board
    #         + empty regions touched only by your stones
    #   White also gets komi.
    # NOTE: dead stones are NOT removed automatically, so play the
    # game out until dead stones are actually captured.
    # -----------------------------------------

    def score(self):
        points = {BLACK: 0, WHITE: 0}
        seen = set()

        for r in range(self.size):
            for c in range(self.size):
                value = self.board[r][c]
                if value != EMPTY:
                    points[value] += 1
                    continue
                if (r, c) in seen:
                    continue

                # flood fill this empty region and record who borders it
                region = {(r, c)}
                borders = set()
                stack = [(r, c)]
                while stack:
                    point = stack.pop()
                    for nr, nc in self.neighbors[point]:
                        v = self.board[nr][nc]
                        if v == EMPTY and (nr, nc) not in region:
                            region.add((nr, nc))
                            stack.append((nr, nc))
                        elif v != EMPTY:
                            borders.add(v)
                seen |= region
                if len(borders) == 1:
                    points[borders.pop()] += len(region)

        return {BLACK: points[BLACK], WHITE: points[WHITE] + self.komi}

    def winner(self):
        s = self.score()
        return BLACK if s[BLACK] > s[WHITE] else WHITE

    # -----------------------------------------
    # Display
    # -----------------------------------------

    def print_board(self):
        print("  " + " ".join(str(c) for c in range(self.size)))
        for r in range(self.size):
            print(r, " ".join(self.board[r]))
        print()

    # ----------------------------------------
    # Influence Calculation (your Phase 2 idea)
    # ----------------------------------------
    # def calc_influence(self, group, liberties):
    #     ...


In [2]:
# ---------- Rule tests: run this cell; it should print "All tests passed" ----------

def setup(rows):
    """Build a game from a picture of the board (list of 9 strings)."""
    g = GoGame()
    for r, line in enumerate(rows):
        for c, ch in enumerate(line.replace(" ", "")):
            g.board[r][c] = ch
    return g

# 1. Capture: White at (0,0) loses its last liberty
g = setup(["W........", "B........"] + ["........."] * 7)
g.current_player = BLACK
res = g.play_move(0, 1)
assert res and res.captured == [(0, 0)] and g.board[0][0] == EMPTY

# 2. Suicide is illegal
g = setup([".B.......", "B........"] + ["........."] * 7)
g.current_player = WHITE
res = g.play_move(0, 0)
assert not res and res.reason == "suicide" and g.board[0][0] == EMPTY

# 3. Suicide that captures is LEGAL
g = setup([".BW......", "BW.......", "W........"] + ["........."] * 6)
g.current_player = WHITE
res = g.play_move(0, 0)   # looks like suicide, but captures (0,1) and (1,0)
assert res and set(res.captured) == {(0, 1), (1, 0)}

# 4. Ko: replay the game from your notebook. B's final recapture at (3,3) was a ko violation.
g = GoGame()
moves = [(4,4),(3,4),(2,4),(4,3),(3,3),(5,3),(3,5),(3,2),(3,6),(2,3),(5,5),(3,4)]
for m in moves:
    assert g.play_move(*m), m
res = g.play_move(3, 3)
assert not res and res.reason == "ko"
g.play_move(6, 6); g.play_move(6, 7)   # play elsewhere ("ko threat"), then retaking is fine
assert g.play_move(3, 3)

# 5. Atari
g = setup(["W........", "B........"] + ["........."] * 7)
assert g.is_in_atari(0, 0) and len(g.groups_in_atari(WHITE)) == 1

# 6. Eyes: a corner group with two real eyes
g = setup([".B.B.....",
           "BBBB.....",
           "........."] + ["........."] * 6)
group = g.get_group(1, 0)
assert g.is_eye(0, 0, BLACK) and g.is_eye(0, 2, BLACK) and g.has_two_eyes(group)
# Neither eye can be filled by White (both moves are suicide)
g.current_player = WHITE
assert not g.is_legal(0, 0) and not g.is_legal(0, 2)

# 7. False eye: enemy stone on the diagonal of an edge point
g = setup(["B.B......", "WB......."] + ["........."] * 7)
assert not g.is_eye(0, 1, BLACK)

# 8. Passing ends the game; empty board -> White wins on komi
g = GoGame()
g.pass_turn(); g.pass_turn()
assert g.game_over() and g.score() == {BLACK: 0, WHITE: 7.5} and g.winner() == WHITE

# 9. Illegal moves don't change anything
g = GoGame(); g.play_move(4, 4)
before = g.board_key()
assert not g.play_move(4, 4) and not g.play_move(9, 9) and g.board_key() == before

# 10. copy() is independent
g = GoGame(); h = g.copy(); h.play_move(4, 4)
assert g.board[4][4] == EMPTY and h.board[4][4] == BLACK

print("All tests passed")


All tests passed


In [3]:
# ---------- Stability + speed check: random games ----------
import random, time

def random_playout(game, max_moves=200):
    """Both players play random legal moves, never filling their own eyes,
    and pass when nothing else is left. (This is the core of Monte Carlo.)"""
    while not game.game_over() and len(game.move_history) < max_moves:
        player = game.current_player
        candidates = [(r, c) for r in range(game.size) for c in range(game.size)
                      if game.board[r][c] == EMPTY and not game.is_eye(r, c, player)]
        random.shuffle(candidates)
        for r, c in candidates:
            if game.play_move(r, c):
                break
        else:
            game.pass_turn()
    return game

random.seed(0)
start = time.time()
n_games = 200
black_wins = 0
for _ in range(n_games):
    g = random_playout(GoGame())
    # sanity: no group on the board may have zero liberties
    assert all(libs for _, _, libs in g.all_groups())
    black_wins += g.winner() == BLACK
elapsed = time.time() - start
print(f"{n_games} random games, no rule violations. "
      f"{n_games/elapsed:.0f} games/sec. Black won {black_wins/n_games:.0%}.")


200 random games, no rule violations. 189 games/sec. Black won 40%.


In [4]:
# ---------- Play a game by hand (type 'quit' to stop) ----------
game = GoGame()

while not game.game_over():
    game.print_board()
    print(f"Current player: {game.current_player}   captures: {game.captures}")
    move_input = input("Enter your move (row col), 'pass', or 'quit': ").strip().lower()

    if move_input == "quit":
        break
    if move_input == "pass":
        game.pass_turn()
        continue

    try:
        row, col = map(int, move_input.split())
    except ValueError:
        print("Invalid input. Please enter 'row col', 'pass', or 'quit'.")
        continue

    result = game.play_move(row, col)
    if not result:
        print(f"Illegal move ({result.reason}), try again.")
    elif result.captured:
        print(f"Captured {len(result.captured)} stone(s)!")

print("Game Over!")
game.print_board()
score = game.score()
print(f"Score (area + komi {game.komi}):  Black {score[BLACK]}  White {score[WHITE]}  ->  {game.winner()} wins")


  0 1 2 3 4 5 6 7 8
0 . . . . . . . . .
1 . . . . . . . . .
2 . . . . . . . . .
3 . . . . . . . . .
4 . . . . . . . . .
5 . . . . . . . . .
6 . . . . . . . . .
7 . . . . . . . . .
8 . . . . . . . . .

Current player: B   captures: {'B': 0, 'W': 0}
Game Over!
  0 1 2 3 4 5 6 7 8
0 . . . . . . . . .
1 . . . . . . . . .
2 . . . . . . . . .
3 . . . . . . . . .
4 . . . . . . . . .
5 . . . . . . . . .
6 . . . . . . . . .
7 . . . . . . . . .
8 . . . . . . . . .

Score (area + komi 7.5):  Black 0  White 7.5  ->  W wins


In [5]:
# Phase 2: Solver Aspect

  # For a 9x9 board every move will impact the whole board. First move will usually always be the center (4 4)
  # For a 11x11 board will allow for slightly more space between edge and corner moves before the fights
  # For a 19x19 board will go corners --> edge --> middle with fights where there is common interest. More long term thinking.

  # To calculate a moves impact on winpct I will need to calculate a function for influence in combination with liberties


In [6]:
# Phase 3: Monte Carlo simulation